# AC SugarSims: Architecture Experiment (Fresh Start)
**18 runs** | 6 conditions x 3 seeds x 3,000 steps | Parallel execution

**Changes from previous run:**
- Innovation is independent of SEVC (vanilla firms can innovate)
- Competition drives innovation (Aghion inverted-U on HHI)
- Corrected min()-based Horizon Index
- Parallel subprocess execution (uses all CPU cores)


In [ ]:
# ── 1. Setup ──
import os, subprocess, multiprocessing

# Hardware
try:
    gpu = subprocess.check_output(['nvidia-smi','--query-gpu=name,memory.total',
                                   '--format=csv,noheader']).decode().strip()
    print(f"GPU: {gpu}")
except: print("No GPU")

cpus = multiprocessing.cpu_count()
print(f"CPUs: {cpus}")
N_WORKERS = max(1, cpus - 1)
print(f"Parallel workers: {N_WORKERS}")


In [ ]:
# ── 2. Clone/Update Repo ──
import os
if os.path.exists('ac_sugarsims'):
    %cd ac_sugarsims
    !git pull
else:
    !git clone https://github.com/caseymrobbins/ac_sugarsims.git
    %cd ac_sugarsims

print(f"\nWorking dir: {os.getcwd()}")
!ls *.py | head -20


In [ ]:
# ── 3. Upload Updated Files ──
# Upload: run_parallel.py, horizon_index.py, innovation.py
from google.colab import files
uploaded = files.upload()
for name, data in uploaded.items():
    with open(name, 'wb') as f:
        f.write(data)
    print(f"Saved: {name} ({len(data)} bytes)")


In [ ]:
# ── 4. Verify Modules ──
import importlib
for m in ['environment','agents','planner','metrics','trust',
          'innovation','sustainable_capitalism','horizon_index',
          'economy','banking']:
    try:
        mod = importlib.import_module(m)
        importlib.reload(mod)  # force reload in case of cached versions
        print(f"  OK: {m}")
    except Exception as e:
        print(f"  FAIL: {m}: {e}")


In [ ]:
# ── 5. Smoke Test ──
from environment import EconomicModel
import time

model = EconomicModel(seed=42, grid_width=40, grid_height=40,
                      n_workers=50, n_firms=5, n_landowners=3,
                      objective="SUM_RAW")
t0 = time.time()
for i in range(100):
    model.step()
elapsed = time.time() - t0

h = model.metrics_history[-1]
print(f"100 steps in {elapsed:.1f}s ({elapsed/100*3000/60:.1f} min estimated for 3000 steps)")
print(f"Workers: {h.get('n_workers','?')}, Firms: {h.get('n_firms','?')}")
print(f"Unemployment: {h.get('unemployment_rate',0):.3f}")
print(f"HI: {h.get('horizon_index',0):.3f}")
print(f"Mean skill: {h.get('mean_skill',0):.3f}")
print("\nSmoke test passed")


In [ ]:
# ── 6. RUN THE EXPERIMENT ──
import time
t0 = time.time()

# Use all available cores minus 1
!python run_parallel.py --workers {N_WORKERS}

elapsed = time.time() - t0
print(f"\nTotal wall time: {elapsed/60:.1f} minutes")


In [ ]:
# ── 7. Load Results ──
import pandas as pd
import os

raw_dir = "results/architecture/raw_data"
if not os.path.exists(raw_dir):
    print("ERROR: No results found. Check the run output above for errors.")
else:
    files = sorted([f for f in os.listdir(raw_dir) if f.endswith('.parquet')])
    print(f"Found {len(files)} result files:")
    for f in files:
        print(f"  {f}")

    all_data = pd.concat([pd.read_parquet(f"{raw_dir}/{f}") for f in files],
                         ignore_index=True)
    print(f"\nTotal rows: {len(all_data)}")
    print(f"Conditions: {sorted(all_data['condition'].unique())}")
    print(f"Seeds: {sorted(all_data['seed'].unique())}")


In [ ]:
# ── 8. Quick Comparison ──
import numpy as np

late = all_data[all_data['step'] >= 2500]
conditions = sorted(late['condition'].unique())

metrics = [
    ('worker_min', 'Worker Floor', 'higher'),
    ('worker_mean', 'Worker Mean', 'higher'),
    ('worker_gini', 'Worker Gini', 'lower'),
    ('unemployment_rate', 'Unemployment', 'lower'),
    ('agency_floor', 'Agency Floor', 'higher'),
    ('n_workers', 'Population', 'higher'),
    ('n_firms', 'Firms', 'higher'),
    ('n_active_cartels', 'Cartels', 'lower'),
    ('mean_skill', 'Mean Skill', 'higher'),
    ('total_production', 'Production', 'higher'),
    ('horizon_index', 'Horizon Index', 'higher'),
    ('mean_firm_floor', 'Firm Floor', 'higher'),
]

print("=" * 120)
print("STEADY STATE (steps 2500-3000, mean across seeds)")
print("=" * 120)

header = f"{'Metric':<18}"
for c in conditions:
    short = c.split('_')[0]
    header += f" {short:>14}"
print(header)
print("-" * 120)

for key, label, direction in metrics:
    if key not in late.columns: continue
    row = f"{label:<18}"
    vals = {}
    for c in conditions:
        v = late[late['condition'] == c][key].mean()
        vals[c] = v
    best = min(vals.values()) if direction == 'lower' else max(vals.values())
    for c in conditions:
        v = vals[c]
        mark = "*" if abs(v - best) < abs(best) * 0.01 + 1e-9 else " "
        if abs(v) >= 1000: row += f" {v:>13,.0f}{mark}"
        elif abs(v) >= 1: row += f" {v:>13.2f}{mark}"
        else: row += f" {v:>13.4f}{mark}"
    print(row)


In [ ]:
# ── 9. Staircase Chart ──
import matplotlib.pyplot as plt

conditions_order = ['C1_vanilla_sum', 'C2_sevc_sum', 'C3_sevc_hi_sum',
                    'C4_sevc_trust_sum', 'C5_sevc_trust_hi_sum', 'C6_full_topo']
labels = ['C1:Vanilla', 'C2:+SEVC', 'C3:+HI', 'C4:+Trust', 'C5:+T+HI', 'C6:TOPO']
colors_list = ['#94a3b8', '#6366f1', '#8b5cf6', '#f59e0b', '#f97316', '#10b981']

stair_metrics = {
    'worker_min': 'Worker Floor',
    'unemployment_rate': 'Unemployment',
    'agency_floor': 'Agency Floor',
    'n_firms': 'Firm Count',
    'n_active_cartels': 'Cartels',
    'mean_skill': 'Mean Skill',
    'total_production': 'Production',
    'horizon_index': 'Horizon Index',
}

fig, axes = plt.subplots(2, 4, figsize=(22, 10))
fig.suptitle('Architecture Staircase: Each Layer Contribution', fontsize=16, fontweight='bold')

for idx, (key, title) in enumerate(stair_metrics.items()):
    if key not in late.columns: continue
    ax = axes[idx // 4][idx % 4]
    means, stds = [], []
    for c in conditions_order:
        by_seed = late[late['condition'] == c].groupby('seed')[key].mean()
        means.append(by_seed.mean())
        stds.append(by_seed.std())
    ax.bar(range(len(means)), means, yerr=stds, color=colors_list, alpha=0.85, capsize=3)
    ax.set_xticks(range(len(labels)))
    ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=8)
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('staircase.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── 10. Download ──
import shutil
shutil.make_archive('architecture_results', 'zip', 'results/architecture')
from google.colab import files
files.download('architecture_results.zip')
print("Done!")
